## Part 1 (5 marks)

Load the four CSV files provided alongside this notebook. They are:
- `child_mortality_0_5_year_olds_dying_per_1000_born.csv` -- child mortality (per 1000 live births)
- `co2_pcap_cons.csv` -- CO2 emissions per capita (in tonnes)
- `gdp_pcap.csv` -- GDP per capita (in 2021 USD)
- `mincpcap_cppp.csv` -- inflation-adjusted average daily income ($/person/day)

The data is formatted similarly to that in Lab 2, but is not identical.

In [1]:
# TODO: load the four CSVs into four pandas DataFrames
import pandas as pd

df1 = pd.read_csv('child_mortality_0_5_year_olds_dying_per_1000_born.csv')
df2 = pd.read_csv('co2_pcap_cons.csv')
df3 = pd.read_csv('gdp_pcap.csv')
df4 = pd.read_csv('mincpcap_cppp.csv')

print(df1.info())
print(df2.info())
print(df3.info())
print(df4.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55536 entries, 0 to 55535
Data columns (total 3 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   country                                            55536 non-null  object 
 1   time                                               55536 non-null  int64  
 2   child_mortality_0_5_year_olds_dying_per_1000_born  55531 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.3+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41255 entries, 0 to 41254
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   country        41255 non-null  object 
 1   time           41255 non-null  int64  
 2   co2_pcap_cons  41252 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 967.0+ KB
None
<class 'pandas.core.frame.Da

## Part 2 (15 marks)

Some entries in the data are invalid -- they are either empty values (indicated by `pandas.DataFrame.isna`), or 0, or negative. Some countries are missing entirely from some datasets.

Write code that cleans up the data. Where values for some years are missing/invalid within a country, each should be replaced by the next valid one. Where a country is missing entirely from one dataset, remove it from the others also.

Then, save the resulting clean data back to new csv files (don't overwrite the old ones!). Add `_clean` to the end of the filename (before `.csv`). You must include these files in your submission for grading.

In [2]:
import numpy as np
print(df1.isna().sum())
print('-----')
print(df2.isna().sum())
print('-----')
print(df3.isna().sum())
print('-----')
print(df4.isna().sum())
print('-----')
#TODO: replacing missing/invalid values with the next valid one
cols1 = df1.select_dtypes(include=[np.number]).columns
df1[cols1] = df1[cols1].mask(df1[cols1] <= 0)
df1 = df1.sort_values(by=['country','time'])
df1[cols1] = df1.groupby('country')[cols1].bfill()

cols2 = df2.select_dtypes(include=[np.number]).columns
df2[cols2] = df2[cols2].mask(df2[cols2] <= 0)
df2 = df2.sort_values(by=['country','time'])
df2[cols2] = df2.groupby('country')[cols2].bfill()

cols3 = df3.select_dtypes(include=[np.number]).columns
df3[cols3] = df3[cols3].mask(df3[cols3] <= 0)
df3 = df3.sort_values(by=['country','time'])
df3[cols3] = df3.groupby('country')[cols3].bfill()

cols4 = df4.select_dtypes(include=[np.number]).columns
df4[cols4] = df4[cols4].mask(df4[cols4] <= 0)
df4 = df4.sort_values(by=['country','time'])
df4[cols4] = df4.groupby('country')[cols4].bfill()
# TODO: remove countries missing from any dataset
common_countries = set(df1['country']) & set(df2['country']) & set(df3['country']) & set(df4['country'])
df1=df1[df1['country'].isin(common_countries)]
df2=df2[df2['country'].isin(common_countries)]
df3=df3[df3['country'].isin(common_countries)]
df4=df4[df4['country'].isin(common_countries)]

# TODO: save the clean data to new csv files
df1.to_csv('child_mortality_0_5_year_olds_dying_per_1000_born_clean.csv',index=False)
df2.to_csv('co2_pcap_cons_clean.csv',index=False)
df3.to_csv('gdp_pcap_clean.csv',index=False)
df4.to_csv('mincpcap_cppp_clean.csv',index=False)

country                                              0
time                                                 0
child_mortality_0_5_year_olds_dying_per_1000_born    5
dtype: int64
-----
country          0
time             0
co2_pcap_cons    3
dtype: int64
-----
country     0
time        0
gdp_pcap    6
dtype: int64
-----
country          0
time             0
mincpcap_cppp    3
dtype: int64
-----


## Part 3 (10 marks)

Use your cleaned data (from Part 2) for this part and all subsequent parts!

For each country, find the first year when its GDP reached 90% of its value in 2010. Your code should then print the mean year (i.e. the average across countries) when this occurred (and not print anything else).

In [3]:
# TODO: find the first year when each country's GDP reached 90% of its value in 2010
ndf1 = pd.read_csv('child_mortality_0_5_year_olds_dying_per_1000_born_clean.csv')
ndf2 = pd.read_csv('co2_pcap_cons_clean.csv')
ndf3 = pd.read_csv('gdp_pcap_clean.csv')
ndf4 = pd.read_csv('mincpcap_cppp_clean.csv')

#find 90% of 2010's gpd of each country in a new sheet
gdp_2010=ndf3[ndf3['time']==2010][['country','gdp_pcap']]
gdp_2010['threshold'] = gdp_2010['gdp_pcap']*0.9
target_data = gdp_2010[['country','threshold']]

#merge the data back using country
merged_df = pd.merge(ndf3,target_data,on='country')
#only retain gpd lower than threshold gpd
qualified = merged_df[merged_df['gdp_pcap']>= merged_df['threshold']]

first_year=qualified.groupby('country')['time'].min()
mean_year=first_year.mean()


# TODO: print the mean year (rounded to the nearest integer)
print(round(mean_year))

1993


## Part 4 (15 marks)

For each country **separately**, train a simple linear regression model (using `sklearn.linear_model.LinearRegression`) to predict the child mortality rate from the GDP. Use *every fifth year* from 1800-2010 (inclusive) as training data, i.e. `1800, 1805, ..., 2005, 2010`. Then use each model to make a prediction for the year 2020.

Save these predictions into a CSV file with one row per country, each row containing the country id and the predicted value. Include a header row with column names `country` and `predicted_child_mortality`. Name the file `predicted_child_mortality.csv`. You must include this file in your submission for grading.

Your code should then print the five countries for which the prediction is closest to the true mortality value in the dataset, i.e. those countries whose respective model works best.

In [4]:
from sklearn.linear_model import LinearRegression

# TODO: for each country, fit a linear regression model to the training data, and make a prediction for 2020
full_data=pd.merge(ndf1,ndf3,on=['country','time'])
train_data=full_data[(full_data['time']>=1800) & (full_data['time']<=2010) & (full_data['time']%5==0)]

all_2020_data=full_data[full_data['time']==2020]
countries=full_data['country'].unique()

result=[]
for country in countries:
    country_data=train_data[train_data['country']==country]
    country_2020=all_2020_data[all_2020_data['country']==country]

    #if this country do not have data, skip rest
    if len(country_data)==0 or len(country_2020)==0:
        continue

    gdp=country_data[['gdp_pcap']]
    mortality=country_data['child_mortality_0_5_year_olds_dying_per_1000_born']
    pre2020=country_2020[['gdp_pcap']]

    lin=LinearRegression()
    lin.fit(gdp,mortality)

    prediction=lin.predict(pre2020)[0]

    true_val=country_2020['child_mortality_0_5_year_olds_dying_per_1000_born'].values[0]

    result.append({'country':country,'prediction_child_mortality':prediction,"true_value":true_val})

# TODO: save the predictions to a csv file
result_df=pd.DataFrame(result)
result_df[['country', 'prediction_child_mortality']].to_csv('predicted_child_mortality.csv')
# TODO: print the five countries with the smallest error
result_df['error'] = abs(result_df['prediction_child_mortality'] - result_df['true_value'])
top5 = result_df.sort_values('error').head(5)
print("top5 closest prediction：", top5['country'].tolist())

#print(result_df)

top5 closest prediction： ['tcd', 'omn', 'sdn', 'mwi', 'jor']


## Part 5 (5 marks)

In Part 4 above, you should have found some countries have *negative* predictions for 2020, which is impossible.  Write 3-4 sentences discussing this -- what happens, and how one might fix it.

You may find it helpful to generate a plot for some country showing the training data and the regression line -- however you are not required to do so (only your answer will be graded, not the plot).

**I think the main reason is the relation between GDP and child mortality is not linear.Over the 200-year history, some of the  countries move from low GDP (where mortality drops fast) to high GDP (where mortality flattens at 0) but it will never go below 0. However, linear regression fits a straght line. It 'remembers' the steep drop from the past and extends it to the present, causing negative predictions for 2020.To fix this, we could take the log of the GDP data before training to make the relationship more linear**